# Week 4 — Part 1: Preprocessing + ISIC-2019 Rare-Class Augmentation
**Team CodeCrafters (AI ML-21) — IEEE EMBS Pune Chapter Student Internship 2026**

**Goal of this notebook:**
1. Load HAM10000 with proper `lesion_id`-level grouping (prevents data leakage across train/val/test)
2. Pull **only non-overlapping** images from ISIC-2019 for the rare classes (`df`, `vasc`, `akiec`, `bcc`) to fix class imbalance
3. Apply hair removal (BlackHat) + CLAHE preprocessing (built into the dataset pipeline, not just visualization — per the Week 2→3 fix)
4. Produce a leak-free, lesion-grouped stratified 70/15/15 split
5. Save the final combined metadata CSV + class distribution plots for the next notebook (training)

**Important note on ISIC-2019:** ISIC-2019 is built FROM HAM10000 (plus BCN_20000 + MSK). Many of its images are literally the same lesions as HAM10000. If we naively merge the two datasets, we'll re-introduce the exact lesion-level leakage bug we already fixed in Week 3. This notebook explicitly de-duplicates by `image_id` so we only add **genuinely new** images.

**Running in Google Colab (CPU-only, no GPU needed):** Run the two **Colab Setup** cells right below first — they install the Kaggle API, download HAM10000 + ISIC-2019 directly, and mount Google Drive so your output CSVs persist after the session ends.


## Colab Setup (run this first — Kaggle-specific cells below have been adjusted to match)
This notebook is CPU-only (no GPU needed) and pulls the datasets directly from Kaggle using the Kaggle API. You'll need a `kaggle.json` API token (Kaggle account -> Settings -> API -> Create New Token).

In [ ]:
# Colab Cell A — Install Kaggle API, upload token, download datasets
!pip install -q kaggle

from google.colab import files
import os

print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token)")
uploaded = files.upload()  # select kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)

# HAM10000
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p {DATA_ROOT}/ham10000 --unzip

# ISIC-2019 (try primary slug, fall back to alternate if it 404s)
import subprocess
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'andrewmvd/isic-2019', '-p', f'{DATA_ROOT}/isic2019', '--unzip'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('Primary ISIC slug failed, trying alternate...')
    !kaggle datasets download -d salviohexia/isic-2019-skin-lesion-images-for-classification -p {DATA_ROOT}/isic2019 --unzip

print('\nDownload complete. Contents:')
print('HAM10000:', os.listdir(f'{DATA_ROOT}/ham10000')[:10])
print('ISIC2019:', os.listdir(f'{DATA_ROOT}/isic2019')[:10] if os.path.exists(f'{DATA_ROOT}/isic2019') else 'NOT FOUND')

In [ ]:
# Colab Cell B — Use local Colab storage instead of Drive
import os

OUTPUT_DIR_COLAB = '/content/week4_preprocessing'
os.makedirs(OUTPUT_DIR_COLAB, exist_ok=True)
print('Outputs will be saved to:', OUTPUT_DIR_COLAB)
print('NOTE: this is temporary Colab storage — it will be deleted when the session ends.')
print('Make sure to run the download cell at the end of this notebook before closing Colab.')

In [ ]:
# Cell 1 — Imports & config
import os, glob, json, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 224

# 7-class taxonomy (must match across HAM10000 and ISIC)
CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# Rare classes we specifically want to boost using ISIC-2019
RARE_CLASSES = ['df', 'vasc', 'akiec', 'bcc']

# ---- Colab paths (set by Colab Cell A above) ----
HAM_ROOT = '/content/data/ham10000'
HAM_ROOT_ALT = '/content/data/ham10000'

ISIC_ROOT_CANDIDATES = [
    '/content/data/isic2019',
]

OUTPUT_DIR = OUTPUT_DIR_COLAB  # set in Colab Cell B above
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Cell 2 — Locate dataset roots and print structure (sanity check before assuming paths)
def find_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

ham_root = find_existing([HAM_ROOT, HAM_ROOT_ALT])
isic_root = find_existing(ISIC_ROOT_CANDIDATES)

print('HAM10000 root:', ham_root)
print('ISIC-2019 root:', isic_root)

if ham_root:
    print('\nHAM10000 contents:', os.listdir(ham_root)[:10])
if isic_root:
    print('\nISIC-2019 contents:', os.listdir(isic_root)[:10])
else:
    print('\n⚠️ ISIC-2019 not found at expected paths. Check Add Data panel, '
          'list /kaggle/input/ below, and update ISIC_ROOT_CANDIDATES.')
    print(os.listdir('/content/data'))


## Step 1 — Load HAM10000 metadata (lesion-grouped)

In [ ]:
# Cell 3 — Load HAM10000 metadata
ham_meta_path = glob.glob(os.path.join(ham_root, '**', 'HAM10000_metadata*.csv'), recursive=True)[0]
ham_df = pd.read_csv(ham_meta_path)

# Map dx -> our 7-class scheme (HAM10000 dx codes already match: akiec,bcc,bkl,df,mel,nv,vasc)
ham_df = ham_df.rename(columns={'dx': 'label'})
ham_df['label'] = ham_df['label'].str.lower()
assert set(ham_df['label'].unique()) == set(CLASSES), 'Unexpected HAM10000 labels found'

# Locate actual image files (HAM10000 images are split across two folders on Kaggle)
img_paths = glob.glob(os.path.join(ham_root, '**', '*.jpg'), recursive=True)
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in img_paths}
ham_df['image_path'] = ham_df['image_id'].map(img_lookup)
missing = ham_df['image_path'].isna().sum()
print(f'HAM10000: {len(ham_df)} rows, {missing} missing image paths (should be 0)')

ham_df['source'] = 'HAM10000'
print(ham_df['label'].value_counts())


## Step 2 — Load ISIC-2019 metadata, keep ONLY non-overlapping rare-class images
ISIC-2019 ground truth is one-hot encoded across `MEL, NV, BCC, AK, BKL, DF, VASC, SCC, UNK`.
We map `AK -> akiec`, drop `SCC` and `UNK` (not in our taxonomy), and **exclude any image_id already present in HAM10000** to avoid leakage.

In [ ]:
# Cell 4 — Load and filter ISIC-2019
isic_df = pd.DataFrame()

if isic_root:
    gt_path = glob.glob(os.path.join(isic_root, '**', '*GroundTruth*.csv'), recursive=True)
    if not gt_path:
        gt_path = glob.glob(os.path.join(isic_root, '**', '*.csv'), recursive=True)
    gt_path = gt_path[0]
    isic_raw = pd.read_csv(gt_path)
    print('ISIC ground truth columns:', isic_raw.columns.tolist())

    # ISIC one-hot columns -> single label column
    isic_label_map = {'MEL': 'mel', 'NV': 'nv', 'BCC': 'bcc', 'AK': 'akiec',
                       'BKL': 'bkl', 'DF': 'df', 'VASC': 'vasc'}  # SCC, UNK intentionally dropped
    available_cols = [c for c in isic_label_map if c in isic_raw.columns]

    def row_label(row):
        for c in available_cols:
            if row[c] == 1.0:
                return isic_label_map[c]
        return None

    isic_raw['label'] = isic_raw.apply(row_label, axis=1)
    isic_raw = isic_raw.dropna(subset=['label'])

    id_col = 'image' if 'image' in isic_raw.columns else isic_raw.columns[0]
    isic_raw = isic_raw.rename(columns={id_col: 'image_id'})

    # --- de-duplicate against HAM10000 (critical leakage fix) ---
    ham_ids = set(ham_df['image_id'])
    before = len(isic_raw)
    isic_raw = isic_raw[~isic_raw['image_id'].isin(ham_ids)]
    print(f'ISIC-2019: removed {before - len(isic_raw)} images overlapping with HAM10000 '
          f'({len(isic_raw)} unique images remain)')

    # --- only keep rare classes (this is an augmentation pass, not a full merge) ---
    isic_raw = isic_raw[isic_raw['label'].isin(RARE_CLASSES)]

    isic_img_paths = glob.glob(os.path.join(isic_root, '**', '*.jpg'), recursive=True)
    isic_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in isic_img_paths}
    isic_raw['image_path'] = isic_raw['image_id'].map(isic_lookup)
    isic_raw = isic_raw.dropna(subset=['image_path'])

    isic_raw['source'] = 'ISIC2019'
    isic_raw['lesion_id'] = 'ISIC_' + isic_raw['image_id'].astype(str)  # no lesion grouping info -> treat as singleton lesions

    isic_df = isic_raw[['image_id', 'lesion_id', 'label', 'image_path', 'source']]
    print('\nISIC-2019 rare-class images added per class:')
    print(isic_df['label'].value_counts())
else:
    print('Skipping ISIC-2019 integration — dataset not found. Proceeding with HAM10000 only.')


## Step 3 — Cap ISIC additions so we boost, not overwhelm, the rare classes

In [ ]:
# Cell 5 — Cap how many ISIC images we add per rare class
# Rule of thumb: bring each rare class up to roughly 25-30% of the 'nv' count, not full parity
# (full parity with synthetic-feeling oversampling tends to hurt generalization more than it helps).
target_caps = {
    'df': 600,
    'vasc': 700,
    'akiec': 900,
    'bcc': 1200,
}

if not isic_df.empty:
    capped_parts = []
    for cls, cap in target_caps.items():
        subset = isic_df[isic_df['label'] == cls]
        if len(subset) > cap:
            subset = subset.sample(cap, random_state=SEED)
        capped_parts.append(subset)
    isic_df_capped = pd.concat(capped_parts, ignore_index=True)
    print('Capped ISIC additions:')
    print(isic_df_capped['label'].value_counts())
else:
    isic_df_capped = isic_df


## Step 4 — Combine datasets and visualize the class balance fix

In [ ]:
# Cell 6 — Combine HAM10000 + capped ISIC-2019 rare-class images
combined_df = pd.concat([
    ham_df[['image_id', 'lesion_id', 'label', 'image_path', 'source']],
    isic_df_capped
], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ham_df['label'].value_counts().reindex(CLASSES).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Before: HAM10000 only')
combined_df['label'].value_counts().reindex(CLASSES).plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('After: HAM10000 + ISIC-2019 rare-class boost')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_balance_before_after.png'), dpi=150)
plt.show()

print(f'\nTotal images: {len(combined_df)} (was {len(ham_df)} before augmentation)')


## Step 5 — Lesion-grouped stratified 70/15/15 split
Using `GroupShuffleSplit` on `lesion_id` so that **no lesion's images appear in more than one split** — this is the leakage-proofing step. We approximate stratification by splitting per class and concatenating (group-aware), since sklearn doesn't have a combined stratified+grouped splitter out of the box.

In [ ]:
# Cell 7 — Leak-free, approximately stratified split
def grouped_split(df, test_size, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx_a, idx_b = next(gss.split(df, groups=df['lesion_id']))
    return df.iloc[idx_a], df.iloc[idx_b]

train_parts, val_parts, test_parts = [], [], []
for cls in CLASSES:
    cls_df = combined_df[combined_df['label'] == cls]
    train_val, test = grouped_split(cls_df, test_size=0.15, seed=SEED)
    train, val = grouped_split(train_val, test_size=0.1765, seed=SEED)  # 0.1765*0.85 ≈ 0.15
    train_parts.append(train); val_parts.append(val); test_parts.append(test)

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

# Sanity check: confirm zero lesion_id overlap across splits
assert set(train_df['lesion_id']) & set(val_df['lesion_id']) == set()
assert set(train_df['lesion_id']) & set(test_df['lesion_id']) == set()
assert set(val_df['lesion_id']) & set(test_df['lesion_id']) == set()
print('✅ No lesion_id leakage across train/val/test')

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print('\nTrain class distribution:')
print(train_df['label'].value_counts())


## Step 6 — Preprocessing pipeline: hair removal (BlackHat) + CLAHE

In [ ]:
# Cell 8 — Preprocessing functions (carried over from Week 3, integrated into the pipeline itself)
def remove_hair(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    inpainted = cv2.inpaint(img_bgr, mask, 1, cv2.INPAINT_TELEA)
    return inpainted

def apply_clahe(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l2 = clahe.apply(l)
    lab2 = cv2.merge((l2, a, b))
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)

def preprocess_image(path, size=IMG_SIZE):
    img = cv2.imread(path)
    img = remove_hair(img)
    img = apply_clahe(img)
    img = cv2.resize(img, (size, size))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Quick visual sanity check on a few samples (one per class)
fig, axes = plt.subplots(2, 7, figsize=(18, 5))
for i, cls in enumerate(CLASSES):
    sample_path = combined_df[combined_df['label'] == cls]['image_path'].iloc[0]
    raw = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
    processed = preprocess_image(sample_path)
    axes[0, i].imshow(raw); axes[0, i].set_title(cls); axes[0, i].axis('off')
    axes[1, i].imshow(processed); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Raw', fontsize=10)
axes[1, 0].set_ylabel('Processed', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'preprocessing_preview.png'), dpi=150)
plt.show()


## Step 7 — Class weights (for loss weighting in training notebook)

In [ ]:
# Cell 9 — Compute class weights from the TRAIN split only
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CLASSES)),
    y=train_df['label'].map(CLASS_TO_IDX).values
)
class_weight_dict = {CLASSES[i]: float(w) for i, w in enumerate(class_weights)}
print('Class weights (train split):')
for c, w in class_weight_dict.items():
    print(f'  {c}: {w:.3f}')

with open(os.path.join(OUTPUT_DIR, 'class_weights.json'), 'w') as f:
    json.dump(class_weight_dict, f, indent=2)


## Step 8 — Save final metadata for the training notebook

In [ ]:
# Cell 10 — Persist everything the training notebook needs
train_df.assign(split='train').to_csv(os.path.join(OUTPUT_DIR, 'train.csv'), index=False)
val_df.assign(split='val').to_csv(os.path.join(OUTPUT_DIR, 'val.csv'), index=False)
test_df.assign(split='test').to_csv(os.path.join(OUTPUT_DIR, 'test.csv'), index=False)

summary = {
    'total_images': len(combined_df),
    'ham10000_images': len(ham_df),
    'isic2019_added': len(isic_df_capped) if not isic_df.empty else 0,
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
    'class_weights': class_weight_dict,
}
with open(os.path.join(OUTPUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved to:', OUTPUT_DIR)
print(json.dumps(summary, indent=2))


## Next notebook (Part 2)
Loads `train.csv` / `val.csv` / `test.csv` + `class_weights.json` from `/kaggle/working/week4_preprocessing/`, builds the `SkinDataset` with this preprocessing baked in, and trains the EfficientNet-B0 + ResNet50 + DenseNet121 ensemble.


## Download your outputs
Run this last — it zips the 4 output files and downloads them to your computer. Send the zip to Subham; he'll unzip it and point his training notebook at these files.

In [ ]:
# Final Cell — Zip and download outputs
import shutil
from google.colab import files

zip_path = '/content/week4_preprocessing_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

print('Files included:', os.listdir(OUTPUT_DIR))
files.download(zip_path + '.zip')